# glcuda diagnostics — glbench only

No llama.cpp. Nothing here needs it, and building it on Kaggle costs 28
minutes and has run out of memory overnight.

This notebook answers three questions about **glcuda's own** performance, all
of which need a real GPU that the development machine does not have:

| # | Question | How |
|---|---|---|
| 1 | Does the disabled r256 GEMM pass parity on hardware? | `cargo test -p glcuda` |
| 2 | Where does prefill time actually go? | `GLCUDA_PROFILE_PREFILL=1` |
| 3 | How much of decode is host-side sampling, not GPU work? | `GLCUDA_PROFILE_DECODE=1` |

### Why each one matters

**(1) A measured 41–43% speedup is switched off.** `glcuda/src/runner.rs`
drives the tensor-core GEMM in 64-row sub-slabs, so a 220-token prompt
re-reads every weight four times. `gl_gemm_mma_q8_r256` (256-row reuse) is
written, shipped in the PTX, and benchmarked 41–43% faster with 96 registers
and no spill — but wiring it produced `CUDA_ERROR_MISALIGNED_ADDRESS` and it
was reverted until its parity test goes green *on hardware*. The comment
records why that never happened: **"the T4 runs only bench + profiler, not
cargo test."** This notebook runs cargo test.

⛔ `glcuda/tests/parity.rs` **skips** rather than fails when no CUDA device is
present, printing `SKIP: no CUDA driver/device on this machine`. A green
summary can therefore mean "nothing touched a GPU" — which is exactly the
state the blocker is already in. The cell below reports a skip as proving
nothing, never as a pass.

**(2) Two prefill candidates, deliberately unranked.** An audit found
attention running `attn_decode_rows` (a decode-shaped kernel applied per
prefill row, with no tensor-core path) and no kernel fusion anywhere
(`rms_norm → quantize → gemm` as three kernels, four `quantize_q8` passes per
layer, roughly 45 launches per layer). A 2026-07-12 profile put the attention
core at 39% of prefill — but that was a different session and a different
model. **This notebook does not rank them.** The bucket split decides.

**(3) glbench's decode number includes work llama.cpp's does not.** Every
token copies full-vocabulary logits to the host, applies a repetition penalty
and samples on the CPU, serialised between graph launches with the GPU idle.
`GLCUDA_PROFILE_DECODE=1` splits GPU from host so the engine's decode rate can
be separated from the sampling pipeline's.

### On reading the profiled runs

Profile mode syncs at phase boundaries. The **totals are inflated by the act
of measuring**; the **split between buckets is the usable result**. The
unperturbed reference numbers come from the plain run in Step 4.

## Step 1 — Config

In [ ]:
# ---- edit these if needed -------------------------------------------------
REPO_URL = "https://github.com/gwenland-org/gwenland-ai.git"
BRANCH   = "glbench-vs-llamacpp"
GH_TOKEN = ""

MODEL_REPO = "https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct-GGUF/resolve/main"
MODEL_FILE = "qwen2.5-0.5b-instruct-q4_k_m.gguf"

GEN_TOKENS = 128
WARMUP     = 3
ITERS      = 10
# --------------------------------------------------------------------------

import os, sys, re, json, time, glob, shutil, subprocess, urllib.request

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
os.makedirs(WORK, exist_ok=True)
REPO_DIR = os.path.join(WORK, "gwenland-ai")
OUT_DIR  = WORK

def sh(cmd, cwd=None, timeout=7200, env=None):
    # stdin=DEVNULL so nothing can sit waiting for a human that is not there.
    e = dict(os.environ)
    if env:
        e.update(env)
    try:
        p = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True,
                           timeout=timeout, env=e, stdin=subprocess.DEVNULL)
        return p.returncode, p.stdout, p.stderr
    except subprocess.TimeoutExpired:
        return 124, "", f"timed out after {timeout}s"
    except Exception as ex:
        return 125, "", f"{type(ex).__name__}: {ex}"

rc, out, _ = sh(["nvidia-smi", "--query-gpu=name,compute_cap,memory.total",
                 "--format=csv,noheader"], timeout=60)
GPU_LINE = out.strip() if rc == 0 else "no nvidia-smi"
GPU_COUNT = len([ln for ln in GPU_LINE.splitlines() if ln.strip()]) if rc == 0 else 0
print(f"gpus : {GPU_COUNT}")
for ln in GPU_LINE.splitlines():
    print(f"       {ln}")

# Kaggle hands out two T4s. glcuda runs on one; pinning keeps every number in
# this notebook comparable to every other, and to the runs already recorded.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
if GPU_COUNT > 1:
    print(f"\n{GPU_COUNT} GPUs visible -- pinned to device 0 "
          f"(CUDA_VISIBLE_DEVICES=0) so results stay comparable")

RUN_META = {
    "date_utc": time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()),
    "gpu": GPU_LINE.replace("\n", " | "),
    "gpu_count": GPU_COUNT,
}
if GPU_COUNT == 0:
    print("\nWARNING: no GPU detected. Every diagnostic below needs one; the "
          "parity suite will SKIP and report nothing.")


## Step 2 — Repo and toolchain

In [ ]:
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    url = REPO_URL.replace("https://", f"https://{GH_TOKEN}@") if GH_TOKEN else REPO_URL
    rc, o, e = sh(["git", "clone", "--depth", "1", "--branch", BRANCH, url, REPO_DIR])
    print((o or e)[-1200:])
else:
    # rc is checked here on purpose. A silently failed fetch leaves a stale
    # checkout, and every measurement below then describes code that is not
    # the code under test -- which is exactly how an A/B once ran eight times
    # against two identical arms.
    rc_f, _, e_f = sh(["git", "fetch", "--depth", "1", "origin", BRANCH], cwd=REPO_DIR)
    rc_r, _, e_r = sh(["git", "reset", "--hard", "FETCH_HEAD"], cwd=REPO_DIR)
    if rc_f != 0 or rc_r != 0:
        print("⛔ REFRESH FAILED — the clone is stale:")
        print("   fetch:", (e_f or "").strip()[:200])
        print("   reset:", (e_r or "").strip()[:200])
    else:
        print("refreshed existing clone")

rc, o, _ = sh(["git", "log", "--oneline", "-1"], cwd=REPO_DIR)
GL_COMMIT = o.strip()
rc, o_r, _ = sh(["git", "rev-parse", "--short", f"origin/{BRANCH}"], cwd=REPO_DIR)
REMOTE_HEAD = o_r.strip()
print("commit :", GL_COMMIT)
if REMOTE_HEAD and not GL_COMMIT.startswith(REMOTE_HEAD[:7]):
    print(f"⛔ the checkout is BEHIND {BRANCH} (remote tip {REMOTE_HEAD}). "
          f"Everything measured below describes older code.")

if shutil.which("cargo") is None:
    os.system("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | "
              "sh -s -- -y --default-toolchain stable --profile minimal")
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]
rc, o, _ = sh(["cargo", "--version"], timeout=180)
print("cargo  :", o.strip())


## Step 3 — Model

In [ ]:
MODEL_PATH = None
for cand in [os.path.join(WORK, MODEL_FILE), os.path.join(REPO_DIR, MODEL_FILE)]:
    if os.path.exists(cand) and os.path.getsize(cand) > 10_000_000:
        MODEL_PATH = cand
        break

if MODEL_PATH is None and os.path.isdir("/kaggle/input"):
    for root, _, files in os.walk("/kaggle/input"):
        for f in files:
            if f.lower().endswith(".gguf") and "0.5b" in f.lower():
                MODEL_PATH = os.path.join(root, f)
                break
        if MODEL_PATH:
            break

if MODEL_PATH is None:
    dest = os.path.join(WORK, MODEL_FILE)
    print(f"downloading {MODEL_REPO}/{MODEL_FILE}")
    rc, o, e = sh(["curl", "-fL", "--retry", "3", "-o", dest,
                   f"{MODEL_REPO}/{MODEL_FILE}"], timeout=1800)
    if rc == 0 and os.path.exists(dest) and os.path.getsize(dest) > 10_000_000:
        MODEL_PATH = dest
    else:
        print("download failed:", e[-400:])

if not MODEL_PATH:
    raise RuntimeError("no model — nothing below can run")
MODEL_BYTES = os.path.getsize(MODEL_PATH)
print(f"model : {MODEL_PATH}")
print(f"size  : {MODEL_BYTES/1e9:.3f} GB")


## Step 4 — Build glbench, then the reference run

The binary is cached in `/kaggle/working` and keyed to the commit, so a saved
session skips the build next time. The reference run below is **unperturbed**:
no profiling env vars, so its numbers are the ones to quote.

In [ ]:
GL_BIN = os.path.join(REPO_DIR, "target", "release", "glbench")
BIN_CACHE = os.path.join(WORK, "glbench-bin-cache")
CACHE_MANIFEST = os.path.join(BIN_CACHE, "manifest.json")

# Reuse only on an exact commit match: a binary from other code is a binary
# measuring other code.
RESTORED = False
if os.path.exists(CACHE_MANIFEST):
    try:
        man = json.load(open(CACHE_MANIFEST, encoding="utf-8"))
        src = os.path.join(BIN_CACHE, "glbench")
        if man.get("commit") == GL_COMMIT and os.path.exists(src):
            os.makedirs(os.path.dirname(GL_BIN), exist_ok=True)
            shutil.copy2(src, GL_BIN)
            os.chmod(GL_BIN, 0o755)
            RESTORED = True
            print("reusing cached glbench binary (same commit)")
        else:
            print(f"cache is for a different commit ({man.get('commit')}); rebuilding")
    except Exception as ex:
        print(f"cache unreadable ({type(ex).__name__}); rebuilding")

BUILD_SECS = 0.0
if not RESTORED:
    print("building glbench (release)...")
    t0 = time.time()
    rc, o, e = sh(["cargo", "build", "--release", "-p", "glbench"],
                  cwd=REPO_DIR, timeout=7200)
    BUILD_SECS = time.time() - t0
    if rc != 0:
        print((o + e)[-5000:])
        raise RuntimeError("glbench build failed")
    print(f"built in {BUILD_SECS/60:.1f} min")
    os.makedirs(BIN_CACHE, exist_ok=True)
    shutil.copy2(GL_BIN, os.path.join(BIN_CACHE, "glbench"))
    json.dump({"commit": GL_COMMIT}, open(CACHE_MANIFEST, "w", encoding="utf-8"))
    print(f"cached to {BIN_CACHE} — save this notebook version to keep it")


In [ ]:
# The reference measurement. No profiling env vars: profile mode syncs at
# phase boundaries and inflates exactly what it reports.
GL_JSON_PATH = os.path.join(OUT_DIR, "glbench_result.json")
GL_CMD = [GL_BIN, "run",
          "--engine", "glcuda",
          "--model", MODEL_PATH,
          "--tokens", str(GEN_TOKENS),
          "--warmup", str(WARMUP),
          "--iters", str(ITERS),
          "--out", GL_JSON_PATH]
print("$ " + " ".join(GL_CMD) + "\n")

t0 = time.time()
rc, GL_STDOUT, GL_STDERR = sh(GL_CMD, cwd=REPO_DIR, timeout=7200)
GL_WALL = time.time() - t0
open(os.path.join(OUT_DIR, "glbench_output.txt"), "w", encoding="utf-8").write(
    GL_STDOUT + "\n===== stderr =====\n" + GL_STDERR)
print(GL_STDOUT[-6000:])
if rc != 0:
    print("\n===== stderr =====\n" + GL_STDERR[-3000:])
    raise RuntimeError(f"glbench --engine glcuda failed (exit {rc})")

GL_JSON = json.load(open(GL_JSON_PATH, encoding="utf-8"))
GL_AN = GL_JSON.get("analysis") or {}
_its = (GL_JSON.get("measurements") or {}).get("iterations") or []
PROMPT_TOKENS = int(_its[0]["prompt_tokens"]) if _its and _its[0].get("prompt_tokens") else None
GL_PRE = (GL_AN.get("prefill_tps") or {})
GL_DEC = (GL_AN.get("decode_tps") or {})
print(f"\nprompt tokens : {PROMPT_TOKENS}")
print(f"prefill       : {GL_PRE.get('mean')} tok/s (median {GL_PRE.get('median')})")
print(f"decode        : {GL_DEC.get('mean')} tok/s (median {GL_DEC.get('median')})")
print(f"wall          : {GL_WALL/60:.1f} min")


## Step 5 — Question 1: does r256 pass parity on hardware?

⛔ Read the verdict, not the summary line. The suite skips without a device,
so `0 failed` can mean nothing ran.

In [ ]:
print("$ cargo test -p glcuda --release -- --nocapture\n")
# --test-threads=1 is mandatory, not tidiness. driver.rs::capture uses
# CU_STREAM_CAPTURE_MODE_GLOBAL, which by design blocks every OTHER thread in
# the process from cuMemFree and from launching on the legacy stream while a
# capture is open. Run in parallel and unrelated tests fail with
# CUDA_ERROR_STREAM_CAPTURE_UNSUPPORTED / _INVALIDATED plus a VRAM-leak
# assertion -- three failures that read as regressions and are not. The same
# tests pass serially.
rc_t, t_out, t_err = sh(["cargo", "test", "-p", "glcuda", "--release",
                         "--", "--test-threads=1", "--nocapture"],
                        cwd=REPO_DIR, timeout=7200)
T_HAY = (t_out or "") + "\n" + (t_err or "")
open(os.path.join(OUT_DIR, "glcuda_tests.txt"), "w", encoding="utf-8").write(T_HAY)

SKIPPED = "SKIP: no CUDA driver/device" in T_HAY

# The verdict comes from libtest's own status line, not merely from any line
# mentioning the test. A failing test prints BOTH `test <name> ... FAILED` and
# a panic line naming the test; taking the last match picked the panic, whose
# text contains neither "ok" nor "FAILED", and the verdict came out "unclear"
# for a plainly failed test.
R256_LINE = None
R256_PANIC = None
for line in T_HAY.splitlines():
    s = line.strip()
    if not s.startswith("test ") or "r256_matches_dequantized_reference" not in s:
        if "r256_matches_dequantized_reference" in s and "panicked" in s:
            R256_PANIC = s
        continue
    if s.endswith(" ok") or s.endswith("FAILED") or " ... " in s:
        R256_LINE = s

# The assertion message is where the actual defect is described.
_detail = re.search(r"^gemm_mma_q8_r256\([^\n]*", T_HAY, re.M)
R256_DETAIL = _detail.group(0).strip() if _detail else None

for ln in T_HAY.splitlines():
    if ln.startswith("test result:") or "SKIP:" in ln:
        print(ln)

if SKIPPED:
    R256_VERDICT = "SKIPPED — no CUDA device; this run proves nothing"
elif R256_LINE and R256_LINE.endswith(" ok"):
    R256_VERDICT = "PASSED on hardware"
elif R256_LINE and "FAILED" in R256_LINE:
    R256_VERDICT = "FAILED on hardware"
elif R256_LINE:
    R256_VERDICT = f"ran, unclear: {R256_LINE}"
else:
    R256_VERDICT = "test not found in output"

print(f"\ngemm_mma_q8_r256_matches_dequantized_reference: {R256_VERDICT}")
if R256_LINE:
    print(f"  {R256_LINE}")
if R256_DETAIL:
    print(f"  {R256_DETAIL}")
if SKIPPED:
    print("  ^ skipped for lack of a device. NOT a pass: 'never run on "
          "hardware' is precisely the blocker.")
elif R256_VERDICT == "PASSED on hardware":
    print("  ^ the parity blocker is cleared. What remains is the "
          "CUDA_ERROR_MISALIGNED_ADDRESS seen when wiring it into the real "
          "prefill scratch — a different bug from the one this test covers.")
elif R256_VERDICT == "FAILED on hardware":
    print("  ^ the kernel computes the wrong answer, so its 41-43% benchmark "
          "measured a wrong result. Note the parity cases only use "
          "out_dim=16, in_dim=64 — no shape this model actually runs is "
          "covered, so a pass would not have proven much either.")


## Step 6 — Question 2: where does prefill time go?

Buckets come from `GLCUDA_PROFILE_PREFILL=1`. The totals are perturbed by the
syncs that produce them; the split is not.

In [ ]:
print("$ GLCUDA_PROFILE_PREFILL=1 glbench run ...\n")
rc_p, p_out, p_err = sh(GL_CMD[:-1] + [os.path.join(OUT_DIR, "glbench_prefill_profile.json")],
                        cwd=REPO_DIR, timeout=3600,
                        env={"GLCUDA_PROFILE_PREFILL": "1"})
P_HAY = (p_out or "") + "\n" + (p_err or "")
open(os.path.join(OUT_DIR, "glcuda_prefill_profile.txt"), "w", encoding="utf-8").write(P_HAY)

PREFILL_BUCKETS = [ln.strip() for ln in P_HAY.splitlines()
                   if re.search(r"(qkv|attn|ffn|gate|down|elt|core|norm|kv)\b", ln, re.I)
                   and re.search(r"\d", ln) and ("ms" in ln or "%" in ln)]

if PREFILL_BUCKETS:
    print("measured prefill buckets:")
    for b in PREFILL_BUCKETS:
        print("   ", b)
    print("\nThe largest bucket decides what to fix. Not the audit's order.")
else:
    print("no per-bucket lines found — reported as UNMEASURED.")
    print("The two audit candidates stay unranked; an unmeasured guess is not "
          "a priority.")
    print("\n--- tail, for diagnosis ---")
    print(P_HAY[-2500:])


## Step 7 — Question 3: how much of decode is host-side sampling?

glcuda's decode loop copies full-vocabulary logits to the host, applies the
repetition penalty and samples on the CPU — all between graph launches, GPU
idle. `llama-bench`'s generation test does none of that (it feeds a random
next token), so any comparison against it is affected by whatever this cell
measures.

In [ ]:
print("$ GLCUDA_PROFILE_DECODE=1 glbench run ...\n")
rc_d, d_out, d_err = sh(GL_CMD[:-1] + [os.path.join(OUT_DIR, "glbench_decode_profile.json")],
                        cwd=REPO_DIR, timeout=3600,
                        env={"GLCUDA_PROFILE_DECODE": "1"})
D_HAY = (d_out or "") + "\n" + (d_err or "")
open(os.path.join(OUT_DIR, "glcuda_decode_profile.txt"), "w", encoding="utf-8").write(D_HAY)

_m = re.search(r"\[decode split\][^\n]*", D_HAY)
DECODE_SPLIT = _m.group(0) if _m else None
DECODE_GPU_MS = DECODE_HOST_MS = DECODE_HOST_SHARE = None
if DECODE_SPLIT:
    print(DECODE_SPLIT)
    _g = re.search(r"GPU\s*([\d.]+) ms/tok", DECODE_SPLIT)
    _h = re.search(r"HOST\s*([\d.]+) ms/tok", DECODE_SPLIT)
    _s = re.search(r"host share\s*(\d+)%", DECODE_SPLIT)
    DECODE_GPU_MS = float(_g.group(1)) if _g else None
    DECODE_HOST_MS = float(_h.group(1)) if _h else None
    DECODE_HOST_SHARE = int(_s.group(1)) if _s else None
    if DECODE_GPU_MS:
        print(f"\nGPU-only decode rate: {1000.0/DECODE_GPU_MS:.1f} tok/s")
        print(f"reference (unperturbed, full pipeline): {GL_DEC.get('mean')} tok/s")
        print("\nThe first is what a forward-pass-only benchmark measures. The "
              "second is what generating text actually costs. Both are real; "
              "they answer different questions.")
else:
    print("no [decode split] line — reported as UNMEASURED, not estimated.")
    print("\n--- tail, for diagnosis ---")
    print(D_HAY[-2000:])


## Step 7b — A/B: the engine switch under test

Set by `AB_VARIANT` / `AB_ENV` in the cell below; currently **r256**, the
256-row prefill GEMM. It reads each weight fragment once per 256 token rows
instead of once per 64, so a 220-token prompt reads the weights once rather
than four times. `runner.rs` picks it per call from `n.div_ceil(256) <
n.div_ceil(64)` — the arithmetic, not a tuned constant.

`GLCUDA_MULTI_STREAM_PREFILL` is deliberately not in the list. It measured
-0.6% here, interleaved and banner-verified, and was parked; re-running a
decided experiment burns runs without buying a decision.

### How the decision is made

⛔ **On unprofiled prefill throughput, and on overlap — not on a percentage.**

This machine drifts 5-8% between sessions, so a fixed bar near that size
decides noise. The rule instead is: the arms must not overlap. If the worst
run of one still beats the best run of the other, the effect is larger than
the spread that produced it. If they overlap, that is reported as overlap and
nothing is concluded.

⛔ **The `down`+`o` bucket is a diagnostic, not the verdict.** It comes from a
profiled run whose phase syncs inflate its totals, and it answers *where* time
went. Reading it as the decision is how an earlier run reported PARK for a
variant whose end-to-end number had moved +6.0% — the 20% bar it applied had
been calibrated for a different hypothesis (block count) and a different
mechanism.

Runs are interleaved, baseline and variant alternating, because a sequential
A/B here once reported a change as "17% faster" that was physically
impossible.

In [ ]:
import statistics

# ⛔ Precondition, checked before spending eight runs on it.
#
# An earlier attempt ran the full A/B against a checkout that predated the
# feature. Both arms executed identical code, the numbers came out at -1.2%,
# and only a banner check at the end caught it. A flag the binary has never
# heard of produces a clean, believable, meaningless result.
# Which engine switch is under test. Baseline is always the empty env.
#
# GLCUDA_MULTI_STREAM_PREFILL is deliberately absent: it measured -0.6% here
# (interleaved, banner-verified) and was parked. Put it back only with a
# reason, not out of completeness.
AB_VARIANT = "r256"
AB_ENV     = {"GLCUDA_R256": "1"}
AB_BANNER  = "r256 prefill GEMM enabled"
AB_MARKER  = ("glcuda/src/kernels/mod.rs", "GLCUDA_R256")

_src = os.path.join(REPO_DIR, *AB_MARKER[0].split("/"))
AB_CODE_PRESENT = os.path.exists(_src) and \
    AB_MARKER[1] in open(_src, encoding="utf-8", errors="replace").read()

if not AB_CODE_PRESENT:
    print(f"⛔ SKIPPING the A/B: this checkout has no {AB_VARIANT} code.")
    print(f"   commit : {GL_COMMIT}")
    print(f"   looked : {_src}")
    print("   Both arms would run the same binary and report a difference of "
          "roughly zero, which is not a result.")
    print("   Update the clone (or delete it and re-run Step 2) first.")

# Four, not two. The first r256 A/B came back +6.0% on a machine recorded
# drifting 5-8% between sessions: an effect the same size as the noise cannot
# be called from two samples per arm.
AB_REPEATS = 4
MS_ENV = AB_ENV

def down_o_ms(hay):
    v = [int(x) for x in re.findall(r"down\+o GEMM\s+(\d+)ms", hay)]
    return statistics.median(v) if v else None

# Every bucket, not just down+o. An earlier A/B watched down+o alone, saw it
# flat, and missed that end-to-end had moved: whatever r256 bought landed
# somewhere the parser was not looking. Buckets are medians across the run's
# iterations, so a cold first iteration does not set them.
BUCKETS = ["qkv", "attn", "ffn", "gate+up GEMM", "down+o GEMM", "elementwise"]

def split_ms(hay):
    out = {}
    for b in BUCKETS:
        v = [int(m) for m in
             re.findall(re.escape(b) + r"\s+(\d+)ms", hay)]
        if v:
            out[b] = statistics.median(v)
    return out

def prefill_tps(path):
    try:
        j = json.load(open(path, encoding="utf-8"))
        return ((j.get("analysis") or {}).get("prefill_tps") or {}).get("mean")
    except Exception:
        return None

def one_run(tag, env, profiled):
    out = os.path.join(OUT_DIR, f"ab_{tag}.json")
    e = dict(env)
    if profiled:
        e["GLCUDA_PROFILE_PREFILL"] = "1"
    rc, o, er = sh(GL_CMD[:-1] + [out], cwd=REPO_DIR, timeout=3600, env=e)
    return (o or "") + "\n" + (er or ""), prefill_tps(out), rc

AB = {"base_plain": [], "ms_plain": [], "base_prof": [], "ms_prof": []}
AB_DISCARDED = []
MS_ACTIVE = None

# One sweep is issued and thrown away before anything is recorded, for both
# arms alike.
#
# Not a post-hoc trim: the first run of a process was the HIGHEST of its arm in
# both arms of an earlier A/B (baseline 1824 then 1677/1688/1711; r256 1907
# then 1809/1820/1837). glbench warms up inside the process, but the process
# itself -- model load, first CUDA context work, GPU clock boost -- is not
# warmed by anything. Left in, that one value was the entire overlap between
# two otherwise cleanly separated arms.
#
# The discarded values are printed, so nothing is hidden by being dropped.
AB_WARMUP = 1

for r in range(AB_REPEATS + AB_WARMUP if AB_CODE_PRESENT else 0):
    for tag, env, prof, key in [
        (f"base_plain_{r}", {}, False, "base_plain"),
        (f"ms_plain_{r}", MS_ENV, False, "ms_plain"),
        (f"base_prof_{r}", {}, True, "base_prof"),
        (f"ms_prof_{r}", MS_ENV, True, "ms_prof"),
    ]:
        hay, tps, rc = one_run(tag, env, prof)
        if AB_BANNER in hay and MS_ACTIVE is None:
            MS_ACTIVE = re.search(r"\[glcuda\][^\n]*" + re.escape(AB_BANNER) + r"[^\n]*",
                                  hay).group(0)
        rec = {"tps": tps, "down_o": down_o_ms(hay), "rc": rc,
               "split": split_ms(hay)}
        if r < AB_WARMUP:
            AB_DISCARDED.append((tag, tps))
            print(f"  {tag:16s} rc={rc}  prefill={tps}  (warmup, discarded)")
        else:
            AB[key].append(rec)
            print(f"  {tag:16s} rc={rc}  prefill={tps}  down+o={rec['down_o']}ms")

print()
if AB_DISCARDED:
    print("discarded warmup sweep:",
          ", ".join(f"{t}={v:.0f}" for t, v in AB_DISCARDED if v))
print(f"{AB_VARIANT} banner:", MS_ACTIVE or "NOT SEEN - the flag never took effect")

# Where the change landed, bucket by bucket.
def bucket_med(key, b):
    v = [d["split"][b] for d in AB[key] if d.get("split", {}).get(b) is not None]
    return statistics.median(v) if v else None

SPLIT_TABLE = []
for b in BUCKETS:
    bb, mm = bucket_med("base_prof", b), bucket_med("ms_prof", b)
    if bb and mm:
        SPLIT_TABLE.append((b, bb, mm, 100.0 * (mm - bb) / bb))
if SPLIT_TABLE:
    print()
    print("  bucket (profiled, median ms)   base    variant   change")
    for b, bb, mm, d in SPLIT_TABLE:
        print(f"  {b:28s} {bb:6.1f}  {mm:7.1f}   {d:+6.1f}%")

def med(key, field):
    v = [d[field] for d in AB[key] if d.get(field) is not None]
    return statistics.median(v) if v else None

B_DO, M_DO = med("base_prof", "down_o"), med("ms_prof", "down_o")
B_TPS, M_TPS = med("base_plain", "tps"), med("ms_plain", "tps")

AB_DOWN_GAIN = AB_TPS_GAIN = None
if B_DO and M_DO:
    AB_DOWN_GAIN = (B_DO - M_DO) / B_DO * 100.0
if B_TPS and M_TPS:
    AB_TPS_GAIN = (M_TPS - B_TPS) / B_TPS * 100.0

print()
print(f"down+o median   : {B_DO} ms -> {M_DO} ms"
      + (f"   ({AB_DOWN_GAIN:+.1f}%)" if AB_DOWN_GAIN is not None else ""))
print(f"prefill tok/s   : {B_TPS} -> {M_TPS}"
      + (f"   ({AB_TPS_GAIN:+.1f}%)" if AB_TPS_GAIN is not None else ""))
print()

# Decide on end-to-end prefill throughput, and decide it non-parametrically.
#
# The old rule was a 20% bar on the down+o bucket, calibrated for the
# multi-stream hypothesis (more blocks -> faster down+o). r256's mechanism is
# different -- fewer weight reads, across every GEMM -- and judging it by
# down+o alone reported PARK for a variant whose end-to-end number had moved
# +6.0%. The bucket answers WHERE; it is not the thing being decided.
#
# A percentage bar is also the wrong shape for this machine. Drift here is
# recorded at 5-8% between sessions, so any fixed threshold near that size
# decides noise. Instead the arms must not overlap: if the worst run of one
# still beats the best run of the other, the effect is bigger than the spread
# that produced it. Overlap is reported as overlap.
def _spread(key, field):
    v = sorted(d[field] for d in AB[key] if d.get(field) is not None)
    return (v[0], v[-1]) if v else (None, None)

B_LO, B_HI = _spread("base_plain", "tps")
M_LO, M_HI = _spread("ms_plain", "tps")

if not AB_CODE_PRESENT:
    AB_VERDICT = "NOT RUN - the checkout predates the feature under test"
elif MS_ACTIVE is None:
    AB_VERDICT = (f"INVALID - the {AB_VARIANT} banner never appeared; the arms "
                  f"may be identical")
elif None in (B_LO, M_LO):
    AB_VERDICT = "UNMEASURED - no prefill throughput in one or both arms"
elif M_LO > B_HI:
    AB_VERDICT = (f"KEEP - {AB_VARIANT}'s worst run ({M_LO:.0f}) beats the "
                  f"baseline's best ({B_HI:.0f}); the arms do not overlap")
elif M_HI < B_LO:
    AB_VERDICT = (f"REGRESSION - {AB_VARIANT}'s best run ({M_HI:.0f}) loses to "
                  f"the baseline's worst ({B_LO:.0f})")
else:
    AB_VERDICT = (f"INCONCLUSIVE - the arms overlap (baseline {B_LO:.0f}-{B_HI:.0f}, "
                  f"{AB_VARIANT} {M_LO:.0f}-{M_HI:.0f}); the median moved "
                  f"{AB_TPS_GAIN:+.1f}%, the size of this machine's own drift")
print("VERDICT:", AB_VERDICT)
if AB_TPS_GAIN is not None and AB_DOWN_GAIN is not None and \
        (AB_TPS_GAIN > 0) != (AB_DOWN_GAIN > 0):
    print("NOTE: the bucket and end-to-end throughput disagree in sign. The "
          "profiled run syncs at phase boundaries and the plain run does not; "
          "trust the plain run for 'is it faster', the buckets for 'where'.")


## Step 7c — Isolate the GEMM: is it block count or K length?

The multi-stream A/B refuted the block-count explanation for `down` being 13x
slower per weight byte than `gate`/`up`. Rather than guess again, this runs the
GEMM alone at three shapes chosen to separate the two candidates:

| shape | blocks | K | role |
|---|---:|---:|---|
| `gate` 4864x896 | 76 | 896 | blocks differ from `o_proj`, K the same |
| `o_proj` 896x896 | 14 | 896 | **the control** |
| `down` 896x4864 | 14 | 4864 | K differs from `o_proj`, blocks the same |

* `o_proj` fast and `down` slow → **K length** is the limit, and split-K is the
  lever.
* Both slow → block count still is, and the multi-stream A/B failed for a
  reason we have not found.

No engine code runs here — this is `glcuda/examples/bench.rs`, the GEMM on
synthetic buffers, away from prefill's other 40 kernels.

In [ ]:
print("building the bench example...")
rc, o, e = sh(["cargo", "build", "--release", "-p", "glcuda", "--example", "bench"],
              cwd=REPO_DIR, timeout=3600)
if rc != 0:
    print((o + e)[-3000:])
    raise RuntimeError("bench example build failed")

BENCH_BIN = os.path.join(REPO_DIR, "target", "release", "examples", "bench")
rc, b_out, b_err = sh([BENCH_BIN], cwd=REPO_DIR, timeout=3600)
B_HAY = (b_out or "") + "\n" + (b_err or "")
open(os.path.join(OUT_DIR, "glcuda_bench.txt"), "w", encoding="utf-8").write(B_HAY)

REUSE_LINES = [ln.strip() for ln in B_HAY.splitlines() if "[gemm-reuse" in ln]
for ln in REUSE_LINES:
    print(ln)

# us/token at the largest tile (n64) is the comparable figure: the same
# per-call weight cost spread over the most rows.
def us_per_tok(label, n=64):
    for ln in REUSE_LINES:
        if label in ln:
            m = re.search(rf"n{n}:\s*([\d.]+)us/tok", ln)
            if m:
                return float(m.group(1))
    return None

GEMM_US = {name: us_per_tok(name) for name in ("gate    .5B", "o_proj  .5B", "down    .5B")}
print()
for k_, v in GEMM_US.items():
    print(f"  {k_}  n64 = {v if v is not None else '—'} us/tok")

G, O, D = GEMM_US["gate    .5B"], GEMM_US["o_proj  .5B"], GEMM_US["down    .5B"]
GEMM_VERDICT = None
if None in (G, O, D):
    GEMM_VERDICT = ("UNMEASURED — one or more .5B rows missing from the bench "
                    "output; the two explanations stay unseparated")
else:
    # Per weight byte, not per call: down's matrix is 5.4x o_proj's.
    d_per_byte = D / 4864.0
    o_per_byte = O / 896.0
    g_per_byte = G / 896.0
    print()
    print(f"  per K-element: gate {g_per_byte*1e3:.3f} | o_proj {o_per_byte*1e3:.3f} "
          f"| down {d_per_byte*1e3:.3f}  (ns/tok per K)")
    k_ratio = d_per_byte / o_per_byte
    blk_ratio = o_per_byte / g_per_byte
    print(f"  K effect      (down / o_proj, same blocks): {k_ratio:.2f}x")
    print(f"  block effect  (o_proj / gate, same K)     : {blk_ratio:.2f}x")
    if k_ratio >= 1.5 and blk_ratio < 1.5:
        GEMM_VERDICT = (f"K LENGTH — down costs {k_ratio:.2f}x per K-element against "
                        f"o_proj at identical block count, while block count alone "
                        f"costs {blk_ratio:.2f}x. Split-K is the lever.")
    elif blk_ratio >= 1.5 and k_ratio < 1.5:
        GEMM_VERDICT = (f"BLOCK COUNT — o_proj costs {blk_ratio:.2f}x against gate at "
                        f"identical K. That contradicts the multi-stream A/B, which "
                        f"means the A/B did not test what it claimed.")
    elif k_ratio >= 1.5 and blk_ratio >= 1.5:
        GEMM_VERDICT = (f"BOTH — K {k_ratio:.2f}x and blocks {blk_ratio:.2f}x. They "
                        f"compound, and neither fix alone closes the gap.")
    else:
        GEMM_VERDICT = (f"NEITHER — K {k_ratio:.2f}x, blocks {blk_ratio:.2f}x. The "
                        f"GEMM in isolation does not reproduce the 13x seen in "
                        f"prefill, so the cost is not in the GEMM and this whole "
                        f"line of attack is wrong.")

print()
print("VERDICT:", GEMM_VERDICT)


## Step 7d — The anomaly: `down` in context

Measured on this machine, the same kernel at the same shape and the same
sub-slab size:

| | in prefill | in the bench | |
|---|---:|---:|---:|
| `gate` | 62.5 us | 57.6 us | **1.09x** |
| `down` | 822.9 us | 179.2 us | **4.59x** |

`gate` behaves identically in both contexts. `down` does not. That difference
is the anomaly, and nothing the GEMM itself does explains it — block count, K
length and weight reuse were each measured and none of them separate the two.

`[ffn-context]` walks from the bench's context to prefill's, one difference at
a time, timing only the GEMM:

| step | added |
|---|---|
| s0 | the call alone, same weights every iteration |
| s1 | + L2 flushed before each call (16 MB of traffic, 4x the T4's L2) |
| s2 | + weights rotated across 24 copies, as 24 layers do |
| s3 | + the elementwise pass that precedes it in prefill |

The flush and the quantize sit inside the timed region, so their solo cost is
measured and subtracted — otherwise the ladder reports the scaffolding.

**`gate` is the control.** A candidate explanation has to break `down` and
leave `gate` alone. A step that moves both is a property of this harness, not
of the anomaly — and a ladder where `down` never climbs means the cause is
none of these four things, which is also an answer.

In [ ]:
# B_HAY is the bench output captured in the previous step. (BENCH_LINES is
# a name from r256_validation.ipynb; using it here was a NameError that
# stopped this cell before it printed anything.)
FFN_LINES = [l.strip() for l in B_HAY.splitlines() if "[ffn-context" in l]
GEMV_LINES = [l.strip() for l in B_HAY.splitlines() if "[gemv-vs-gemm" in l]
for l in GEMV_LINES:
    print(l)

# GEMV buys 8x the blocks and pays ntok weight reads instead of one. Under
# 1.00x it is worth it anyway, which would mean occupancy dominates and
# split-K -- the same blocks without the traffic -- is the fix.
GEMV_RATIO = {}
for l in GEMV_LINES:
    m = re.search(r"\[gemv-vs-gemm (\w+)", l)
    r = re.search(r"GEMV/GEMM ([\d.]+)x", l)
    if m and r:
        GEMV_RATIO[m.group(1)] = float(r.group(1))
if GEMV_RATIO:
    print()
    for name, ratio in GEMV_RATIO.items():
        print(f"  {name:6s} GEMV/GEMM = {ratio:.2f}x"
              + ("  <- more blocks beat more traffic; split-K is the fix"
                 if ratio < 1.0 else "  <- traffic wins; the GEMM shape is not the problem"))
print()
for l in FFN_LINES:
    print(l)

def ladder(label):
    for l in FFN_LINES:
        if f"[ffn-context {label}" in l:
            return {int(m.group(1)): float(m.group(2))
                    for m in re.finditer(r"s(\d) ([\d.]+)us", l)}
    return {}

G, D = ladder("gate"), ladder("down")
PREFILL_DOWN_US = 822.9          # measured, one 64-row sub-slab
FFN_VERDICT, FFN_TABLE = None, []

if not (G and D) or 0 not in G or 0 not in D:
    FFN_VERDICT = ("UNMEASURED - the ladder produced no usable lines for one "
                   "or both shapes")
else:
    print()
    print("  step   gate            down            differential")
    for s in sorted(set(G) & set(D)):
        gp = 100.0 * (G[s] - G[0]) / G[0]
        dp = 100.0 * (D[s] - D[0]) / D[0]
        FFN_TABLE.append((s, G[s], gp, D[s], dp, dp - gp))
        print(f"  s{s}     {G[s]:7.1f}us {gp:+6.0f}%   {D[s]:7.1f}us {dp:+6.0f}%   "
              f"{dp - gp:+7.0f} pts")

    # The anomaly is the step that separates the two ladders, so the signal is
    # the JUMP in the differential, not its level.
    jumps = [(FFN_TABLE[i][5] - FFN_TABLE[i - 1][5], FFN_TABLE[i][0])
             for i in range(1, len(FFN_TABLE))]
    best_jump, best_step = max(jumps) if jumps else (0.0, None)
    reached = 100.0 * D[max(D)] / PREFILL_DOWN_US

    print()
    print(f"  down at the last step reaches {reached:.0f}% of prefill's "
          f"{PREFILL_DOWN_US}us")

    if best_step is not None and best_jump >= 50.0:
        FFN_VERDICT = (f"s{best_step} is the anomaly: it costs `down` "
                       f"{best_jump:+.0f} points more than it costs `gate`. "
                       f"The ladder reaches {reached:.0f}% of prefill's figure.")
    elif reached < 50.0:
        FFN_VERDICT = (f"NOT REPRODUCED - the ladder only reaches "
                       f"{reached:.0f}% of prefill's {PREFILL_DOWN_US}us, and no "
                       f"step separates `down` from `gate` by more than "
                       f"{best_jump:.0f} points. The cause is none of these "
                       f"four differences, and the next place to look is "
                       f"outside the FFN tail.")
    else:
        FFN_VERDICT = (f"NO SINGLE STEP - `down` reaches {reached:.0f}% of "
                       f"prefill's figure but no step separates it from `gate` "
                       f"by more than {best_jump:.0f} points, so the cost "
                       f"accumulates rather than arriving at one boundary.")

print()
print("VERDICT:", FFN_VERDICT)


## Step 8 — Write `GLCUDA_DIAGNOSTICS.md`

In [ ]:
NL = "\n"
B = []
def w(s=""):
    B.append(s)

w("# glcuda diagnostics")
w()
w(f"- **Date (UTC):** {RUN_META['date_utc']}")
w(f"- **GPU:** {RUN_META['gpu']}"
  + (f" — {RUN_META['gpu_count']} present, pinned to device 0" if RUN_META['gpu_count'] > 1 else ""))
w(f"- **Commit:** `{GL_COMMIT}`")
w(f"- **Model:** `{os.path.basename(MODEL_PATH)}`, {MODEL_BYTES/1e9:.3f} GB")
w(f"- **Workload:** {PROMPT_TOKENS} prompt tokens, {GEN_TOKENS} generated, "
  f"{ITERS} iterations, {WARMUP} warmup")
w()
w("## Reference measurement (unperturbed)")
w()
w("| Phase | mean | median | min | max | std | ±95% CI |")
w("|---|---:|---:|---:|---:|---:|---:|")
for name, s in (("prefill", GL_PRE), ("decode", GL_DEC)):
    w("| {} | {} | {} | {} | {} | {} | {} |".format(
        name,
        *[("{:.1f}".format(s[k]) if isinstance(s.get(k), (int, float)) else "—")
          for k in ("mean", "median", "min", "max", "std_dev", "ci95")]))
w()
_eff = GL_AN.get("ceiling_efficiency")
w(f"Bottleneck verdict: `{GL_AN.get('bottleneck')}`"
  + (f", {_eff*100:.0f}% of the bandwidth ceiling." if isinstance(_eff, (int, float))
     else ", no ceiling efficiency reported."))
w()
w("> The bottleneck label is a threshold on the ceiling fraction, not an "
  "observation of kernel launches — `bottleneck::classify` returns "
  "`launch_overhead` for anything under 40%. glcuda already replays a captured "
  "CUDA graph per token, so read that label as \"far from the ceiling\", not "
  "as a diagnosis.")
w()

# ---- Q1 -------------------------------------------------------------------
w("## 1. r256 GEMM parity on hardware")
w()
w(f"The MMA GEMM runs in 64-row sub-slabs, so a {PROMPT_TOKENS}-token prompt "
  f"re-reads every weight four times. `gl_gemm_mma_q8_r256` (256-row reuse) is "
  f"written, is in the shipped PTX, and benchmarks 41–43% faster at kernel "
  f"level. It is disabled pending this test passing on a GPU.")
w()
w(f"**Verdict: {R256_VERDICT}**")
if R256_LINE:
    w()
    w("```")
    w(R256_LINE)
    w("```")
w()
if SKIPPED:
    w("`glcuda/tests/parity.rs` skips rather than fails without a device, so "
      "the suite's green summary here means nothing ran. That is the same "
      "green summary the blocked state already produces.")
elif R256_VERDICT == "PASSED on hardware":
    w("The parity blocker is cleared. What remains is the "
      "`CUDA_ERROR_MISALIGNED_ADDRESS` that appeared when wiring r256 into the "
      "real prefill scratch — a different bug from the one this test covers, "
      "and one the isolated kernel bench cannot reproduce because it feeds "
      "synthetic buffers.")
elif R256_VERDICT == "FAILED on hardware":
    if R256_DETAIL:
        w("```")
        w(R256_DETAIL)
        w("```")
        w()
    w("The kernel does not match the dequantised reference on this device, so "
      "the 41–43% figure describes a kernel computing the wrong answer. That "
      "is not a speedup, and the `CUDA_ERROR_MISALIGNED_ADDRESS` seen when "
      "wiring it is more likely a symptom of the same indexing defect than a "
      "separate alignment problem.")
    w()
    w("⛔ **The parity cases cover only `out_dim=16, in_dim=64`.** That is one "
      "block with two of eight warps in range — a configuration this model "
      "never runs. Neither GEMM test covers a real shape (896x896, 896x4864, "
      "4864x896), so a pass would not have proven much more than this failure "
      "does. Adding a real-shape case is cheap and worth more than debugging "
      "the toy one.")
w()

# ---- Q2 -------------------------------------------------------------------
w("## 2. Where prefill time goes")
w()
if PREFILL_BUCKETS:
    w("Measured on this model and this device:")
    w()
    w("```")
    for b in PREFILL_BUCKETS:
        w(b)
    w("```")
    w()
    w("Profile mode syncs at phase boundaries, so the totals are inflated by "
      "the measurement. The split is the usable part.")
else:
    w("**Unmeasured.** No per-bucket lines appeared in the profiled run.")
w()
w("The audit's two candidates:")
w()
w("- **Attention** — `attn_decode_rows`, a decode-shaped kernel applied per "
  "prefill row, with no tensor-core path at all. A 2026-07-12 profile put the "
  "attention core at 39% of prefill, on a different session and model.")
w("- **Fusion** — none. `rms_norm → quantize → gemm` are three separate "
  "kernels, there are four `quantize_q8` passes per layer, and roughly 45 "
  "launches per layer (about 1,080 for 24 layers), each re-reading the "
  "activation tensor from global memory.")
w()
w("**Not ranked here.** Which one to write is decided by the split above. "
  "Estimates in this repository have missed by 5.4x and this machine drifts "
  "between sessions, so an audit's discovery order is not evidence.")
w()

# ---- Q3 -------------------------------------------------------------------
w("## 3. Decode: GPU work vs host-side sampling")
w()
w("Every decode token copies full-vocabulary logits to the host, applies the "
  "repetition penalty and samples on the CPU, serialised between graph "
  "launches with the GPU idle.")
w()
if DECODE_SPLIT:
    w("```")
    w(DECODE_SPLIT)
    w("```")
    w()
    if DECODE_GPU_MS and isinstance(GL_DEC.get("mean"), (int, float)):
        w("| | tok/s | what it answers |")
        w("|---|---:|---|")
        w(f"| GPU only | {1000.0/DECODE_GPU_MS:.1f} | what a forward-pass-only "
          f"benchmark measures |")
        w(f"| Full pipeline | {GL_DEC['mean']:.1f} | what generating text "
          f"actually costs |")
        w()
        w("Both are real; they answer different questions. Any comparison "
          "against a benchmark whose generation loop feeds a random next token "
          "— `llama-bench`'s does — is comparing against the first row while "
          "glbench reports the second.")
        w()
    if DECODE_HOST_SHARE is not None:
        w(f"Host share: **{DECODE_HOST_SHARE}%**. "
          + ("Large enough that overlapping sampling with the next token's "
             "graph launch is a real lever, and it is not a kernel change."
             if DECODE_HOST_SHARE >= 20 else
             "Small, so the decode gap is not mostly sampling and kernel work "
             "is the place to look."))
        w()
    w("This run syncs after every token, so its absolute figures are perturbed "
      "by the measurement. The ratio is the usable part; the reference table "
      "at the top is the unperturbed throughput.")
else:
    w("**Unmeasured.** No `[decode split]` line appeared, so the size of the "
      "sampling overhead is not known from this run — not estimated.")
w()

w(f"## 4. A/B: `{AB_VARIANT}`")
w()
w(f"Baseline against `{list(AB_ENV)[0]}=1`, everything else held fixed.")
w()
w(f"Runs are **interleaved**, {AB_REPEATS} repeats per arm. This machine has "
  f"been recorded drifting 5-8% between sessions, and a sequential A/B here "
  f"once reported a physically impossible result.")
w()
w("| | baseline | " + AB_VARIANT + " | change |")
w("|---|---:|---:|---:|")
w("| prefill (tok/s, unprofiled) | {} | {} | {} |".format(
    f"{B_TPS:.1f}" if B_TPS else "-",
    f"{M_TPS:.1f}" if M_TPS else "-",
    f"{AB_TPS_GAIN:+.1f}%" if AB_TPS_GAIN is not None else "-"))
w("| &nbsp;&nbsp;range across runs | {} | {} | |".format(
    f"{B_LO:.0f} - {B_HI:.0f}" if B_LO else "-",
    f"{M_LO:.0f} - {M_HI:.0f}" if M_LO else "-"))
w("| `down`+`o` bucket (ms, profiled) | {} | {} | {} |".format(
    B_DO if B_DO is not None else "-",
    M_DO if M_DO is not None else "-",
    f"{AB_DOWN_GAIN:+.1f}%" if AB_DOWN_GAIN is not None else "-"))
w()
w(f"Engine banner: `{MS_ACTIVE}`" if MS_ACTIVE else
  f"⛔ The `{AB_BANNER}` banner never appeared, so the two arms may have run "
  f"identical code. Treat the comparison as invalid.")
w()
w(f"**Verdict: {AB_VERDICT}**")
w()
w("The decision is made on unprofiled prefill throughput, and on whether the "
  "arms' **ranges overlap** rather than on a percentage bar. A fixed threshold "
  "near this machine's own drift would be deciding noise. The `down`+`o` "
  "bucket is kept as a diagnostic — it answers *where* time went, from a "
  "profiled run whose phase syncs inflate its totals — but it is not what is "
  "being decided, and reading it as the verdict is how an earlier run reported "
  "PARK for a variant whose end-to-end number had moved.")
w()

w("## 5. Isolating the GEMM: block count or K length?")
w()
w("`down` costs about 13x per weight byte what `gate`/`up` cost, on matrices "
  "of identical size (transposes of each other). Two explanations fit, and "
  "section 4 ruled one of them out. This runs the GEMM alone, on synthetic "
  "buffers, at three shapes chosen to separate them:")
w()
w("| shape | blocks | K | role |")
w("|---|---:|---:|---|")
w("| `gate` 4864x896 | 76 | 896 | blocks differ from `o_proj`, K the same |")
w("| `o_proj` 896x896 | 14 | 896 | the control |")
w("| `down` 896x4864 | 14 | 4864 | K differs from `o_proj`, blocks the same |")
w()
if REUSE_LINES:
    w("```")
    for ln in REUSE_LINES:
        w(ln)
    w("```")
    w()
if None not in (G, O, D):
    w("| shape | us/tok (n64) | per K-element (ns/tok) |")
    w("|---|---:|---:|")
    w(f"| `gate` | {G:.2f} | {G/896.0*1e3:.3f} |")
    w(f"| `o_proj` | {O:.2f} | {O/896.0*1e3:.3f} |")
    w(f"| `down` | {D:.2f} | {D/4864.0*1e3:.3f} |")
    w()
w(f"**Verdict: {GEMM_VERDICT}**")
w()
w("Normalising per K-element is what makes the three comparable: `down`'s "
  "matrix is 5.4x `o_proj`'s along K, so a raw per-call time would show it "
  "losing for doing more work rather than for doing it worse.")
w()

w("## 6. The anomaly: `down` in context")
w()
w("The same kernel, the same shape, the same 64-row sub-slab:")
w()
w("| | in prefill | in the bench | |")
w("|---|---:|---:|---:|")
w("| `gate` | 62.5 us | 57.6 us | **1.09x** |")
w("| `down` | 822.9 us | 179.2 us | **4.59x** |")
w()
w("`gate` behaves identically in both contexts and `down` does not, which is "
  "the anomaly. Block count, K length and weight reuse were each measured and "
  "none of them separate the two, so `[ffn-context]` walks from the bench's "
  "context to prefill's one difference at a time, timing only the GEMM. The "
  "flush and quantize costs are measured alone and subtracted.")
w()
if FFN_LINES:
    w("```")
    for l in FFN_LINES:
        w(l)
    w("```")
    w()
if FFN_TABLE:
    w("| step | added | `gate` | `down` | differential |")
    w("|---|---|---:|---:|---:|")
    _added = {0: "the call alone", 1: "+ L2 flush",
              2: "+ 24 rotating weight copies", 3: "+ the preceding quantize"}
    for s, gu, gp, du, dp, diff in FFN_TABLE:
        w(f"| s{s} | {_added.get(s, '')} | {gu:.1f} us ({gp:+.0f}%) | "
          f"{du:.1f} us ({dp:+.0f}%) | {diff:+.0f} pts |")
    w()
w(f"**Verdict: {FFN_VERDICT}**")
w()
w("`gate` is the control, not decoration: a step that slows both shapes is a "
  "property of this harness, and only a step that separates them can explain "
  "why one kernel is 4.6x slower in prefill while the other is not. A ladder "
  "where `down` never climbs is a real answer too — it would mean the cost is "
  "none of these four differences.")
w()

w("## Appendix: raw output")
w()
for title, body in [("glbench — reference run", GL_STDOUT),
                    ("cargo test -p glcuda", T_HAY),
                    ("GLCUDA_PROFILE_PREFILL=1", P_HAY),
                    ("GLCUDA_PROFILE_DECODE=1", D_HAY),
                    ("glcuda bench (GEMM isolation)", B_HAY)]:
    w(f"### {title}")
    w()
    w("```")
    w((body or "(empty)").strip()[:12000])
    w("```")
    w()

MD_PATH = os.path.join(OUT_DIR, "GLCUDA_DIAGNOSTICS.md")
open(MD_PATH, "w", encoding="utf-8").write(NL.join(B))
print(f"wrote {MD_PATH} ({len(NL.join(B))} chars)")


## Step 9 — Summary

In [ ]:
print("=" * 66)
print("glcuda diagnostics")
print("=" * 66)
print(f"gpu            : {RUN_META['gpu']}")
print(f"commit         : {GL_COMMIT}")
print(f"prefill        : {GL_PRE.get('mean')} tok/s")
print(f"decode         : {GL_DEC.get('mean')} tok/s  (full pipeline)")
if DECODE_GPU_MS:
    print(f"decode GPU-only: {1000.0/DECODE_GPU_MS:.1f} tok/s"
          + (f"  (host share {DECODE_HOST_SHARE}%)" if DECODE_HOST_SHARE is not None else ""))
print()
print(f"Q1 r256 parity : {R256_VERDICT}")
print(f"Q2 prefill     : {len(PREFILL_BUCKETS)} bucket lines"
      if PREFILL_BUCKETS else "Q2 prefill     : UNMEASURED")
print(f"Q3 decode split: {'measured' if DECODE_SPLIT else 'UNMEASURED'}")
print()
print("Next step is whichever of Q1/Q2 the numbers above justify — this "
      "notebook deliberately does not choose for you.")
print(f"\nreport: {os.path.join(OUT_DIR, 'GLCUDA_DIAGNOSTICS.md')}")
